### BERT fine-tuning for document classification

In [ ]:
import os
import re
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix

import transformers
from transformers import BertTokenizer, BertModel

import torch
from torch import cuda
from tqdm.auto import tqdm

# Device: CUDA > MPS > CPU
if cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'

print(f'Using device: {device}')
device

- use X.txt and YL1.txt

In [ ]:
X = [line.strip() for line in open('X.txt', encoding='utf-8').readlines()]
y = [int(line.strip()) for line in open('YL1.txt', encoding='utf-8').readlines()]

len(X), len(y), max(y)

Data Cleaning

In [ ]:
def clean_text(text):
    text = re.sub(r"@[A-Za-z0-9]+", ' ', text)
    text = re.sub(r"https?://[A-Za-z0-9./]+", ' ', text)
    text = re.sub(r"[^a-zA-z.!?'0-9]", ' ', text)
    text = re.sub('\t', ' ',  text)
    text = re.sub(r" +", ' ', text)
    return text.lower()

In [ ]:
X = [clean_text(text) for text in X]

### Stratified train/test split with reproducible shuffling

In [ ]:
from sklearn.model_selection import train_test_split

# Use stratified split to keep class distributions aligned between train and test
test_size = 985 / len(X)
train_X, test_X, train_y, test_y = train_test_split(
    X, y,
    test_size=test_size,
    stratify=y,
    random_state=42,
    shuffle=True
)

train_X = np.array(train_X)
train_y = np.array(train_y)
test_X = np.array(test_X)
test_y = np.array(test_y)

len(train_X), len(train_y), len(test_X), len(test_y)

In [ ]:

labels = {
    0: 'Computer Science',
    1: 'Electrical Engineering',
    2: 'Psychology',
    3: 'Mechanical Engineering',
    4: 'Civil Engineering',
    5: 'Medical Science',
    6: 'Biochemistry'
}

len(labels)

class MultiLabelDataset(torch.utils.data.Dataset):

    def __init__(self, text, labels, tokenizer, max_len):
        self.tokenizer = tokenizer
        self.text = text
        self.targets = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.text)

    def __getitem__(self, index):
        text = self.text[index]
        inputs = self.tokenizer(
            text,
            None,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_token_type_ids=True
        )
        ids = inputs['input_ids']
        mask = inputs['attention_mask']
        token_type_ids = inputs["token_type_ids"]

        return {
            'ids': torch.tensor(ids, dtype=torch.long),
            'mask': torch.tensor(mask, dtype=torch.long),
            'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
            'targets': torch.tensor(self.targets[index], dtype=torch.long)
        }

In [ ]:
class MultiLabelDataset(torch.utils.data.Dataset):

  def __init__(self, text, labels, tokenizer, max_len):
      self.tokenizer = tokenizer
      self.text = text
      self.targets = labels
      self.max_len = max_len

  def __len__(self):
      return len(self.text)

  def __getitem__(self, index):
      text = self.text[index]
      inputs = self.tokenizer(
          text,
          None,
          add_special_tokens=True,
          max_length=self.max_len,
          padding='max_length',
          truncation=True,
          return_token_type_ids=True
      )
      ids = inputs['input_ids']
      mask = inputs['attention_mask']
      token_type_ids = inputs["token_type_ids"]

      return {
          'ids': torch.tensor(ids, dtype=torch.long),
          'mask': torch.tensor(mask, dtype=torch.long),
          'token_type_ids': torch.tensor(token_type_ids, dtype=torch.long),
          'targets': torch.tensor(self.targets[index], dtype=torch.long)
      }

**BERT Class**

In [ ]:
class BERTClass(torch.nn.Module):
    def __init__(self, NUM_OUT):
        super(BERTClass, self).__init__()
        self.l1 = BertModel.from_pretrained("bert-base-uncased")
        self.classifier = torch.nn.Linear(768, NUM_OUT)

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        output = self.classifier(pooler)
        return output

**Helper Functions**

In [ ]:
def loss_fn(outputs, targets):
    return torch.nn.CrossEntropyLoss()(outputs, targets)

def train(model, training_loader, optimizer):
    model.train()
    total_loss = 0
    for data in tqdm(training_loader):
        ids = data['ids'].to(device, dtype=torch.long)
        mask = data['mask'].to(device, dtype=torch.long)
        token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
        targets = data['targets'].to(device, dtype=torch.long)

        outputs = model(ids, mask, token_type_ids)
        loss = loss_fn(outputs, targets)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(training_loader)

def validation(model, testing_loader):
    model.eval()
    fin_targets = []
    fin_outputs = []
    with torch.no_grad():
        for data in tqdm(testing_loader):
            targets = data['targets']
            ids = data['ids'].to(device, dtype=torch.long)
            mask = data['mask'].to(device, dtype=torch.long)
            token_type_ids = data['token_type_ids'].to(device, dtype=torch.long)
            outputs = model(ids, mask, token_type_ids)
            outputs = torch.softmax(outputs, dim=1).cpu().detach()
            fin_outputs.append(outputs)
            fin_targets.append(targets)

    return torch.cat(fin_outputs), torch.cat(fin_targets)

def report_metrics(y_true, y_pred, label_names):
    '''Report precision, recall, F1, and confusion matrix.'''
    print("\n" + "="*70)
    print("Classification Report")
    print("="*70)
    print(classification_report(y_true, y_pred, target_names=[label_names[i] for i in range(len(label_names))]))

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(y_true, y_pred, average='macro')
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')

    print(f"\nMacro-averaged:   Precision={precision_macro:.4f}, Recall={recall_macro:.4f}, F1={f1_macro:.4f}")
    print(f"Weighted-average: Precision={precision_weighted:.4f}, Recall={recall_weighted:.4f}, F1={f1_weighted:.4f}")
    print("="*70)

    return {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision_macro': precision_macro,
        'recall_macro': recall_macro,
        'f1_macro': f1_macro,
        'precision_weighted': precision_weighted,
        'recall_weighted': recall_weighted,
        'f1_weighted': f1_weighted
    }

**The Tokenizer**

- Convert a raw string to the ids, masks, and token_type_ids

In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

print(X[0][:200])

# print(tokenizer.encode_plus(
#     X[0],
#     None,
#     add_special_tokens=True,
#     max_length=128,
#     padding='max_length',
#     truncation=True,
#     return_token_type_ids=True
# ))

**Training Setup**
- hyperparameters
- setup dataset
- setup parameters
- setup dataloader

In [ ]:
MAX_LEN = 128
BATCH_SIZE = 8
EPOCHS = 3
NUM_OUT = 7
LEARNING_RATE = 2e-05

training_data = MultiLabelDataset(train_X, train_y, tokenizer, MAX_LEN)
test_data = MultiLabelDataset(test_X, test_y, tokenizer, MAX_LEN)

train_params = {'batch_size': BATCH_SIZE,
                'shuffle': True,
                'num_workers': 0
                }

test_params = {'batch_size': BATCH_SIZE,
                'shuffle': False,
                'num_workers': 0
                }

training_loader = torch.utils.data.DataLoader(training_data, **train_params)
testing_loader = torch.utils.data.DataLoader(test_data, **test_params)

print(f'Training samples: {len(training_data)}, Testing samples: {len(test_data)}')

**Full Dataset Fine-tuning**

In [ ]:
model = BERTClass(NUM_OUT)
model.to(device)

optimizer = torch.optim.Adam(params=model.parameters(), lr=LEARNING_RATE)

for epoch in range(EPOCHS):
    loss = train(model, training_loader, optimizer)
    print(f'Epoch: {epoch}, Loss: {loss:.4f}')
    guess, targs = validation(model, testing_loader)
    guesses = torch.max(guess, dim=1)

    metrics = report_metrics(targs.numpy(), guesses.indices.numpy(), labels)
    print(f"  Accuracy: {metrics['accuracy']:.4f}, Macro-F1: {metrics['f1_macro']:.4f}\n")

**What does the BERT tokenizer do?**

BERT tokenizer converts raw text into the format that the BERT model can understand. Here's what it does:
- **Tokenization**: splits input into smaller units called tokens. BERT uses WordPiece tokenization, which can break down words into subword components if the full word isn't in the vocabulary.
- **Mapping to IDs**: maps tokens to a unique numerical identifier based on BERT's vocabulary, converting text into a sequence of numbers (e.g., "hello" → 7592, "world" → 2088).
- **Adding special tokens**: inserts special tokens that the BERT model requires such as `[CLS]` (indicates the beginning of input), `[SEP]` (separates sentence pairs or ends a sequence), `[PAD]`, and `[MASK]` (for masked language modeling).
- **Creating attention masks and token type IDs**: Attention masks indicate which tokens should be attended to, and Token Type IDs help the model differentiate between segments.

**Loss Function**

Since the problem is multi-class classification - where each input belongs to exactly one class — we use CrossEntropyLoss as the loss function. The reasons are:
- It uses softmax internally to ensure that all class probabilities add up to 1, preventing the model from giving high scores to multiple classes at once.
- The loss is calculated by taking the negative log of the probability of the correct class, making the computation simple and helping the model adjust its weights properly.
- CrossEntropyLoss is based on likelihood principles, which means the model learns in a statistically sound way, often leading to faster learning since weight updates match the size of errors across all classes.

### Why Fine-tuning is Powerful

Fine-tuning leverages a pre-trained model's broad language knowledge for a specific task. Instead of training from scratch, you adjust the model with examples from your own task. This approach:
- Requires less task-specific training data than training from scratch
- Takes less time and compute than training a model from the ground up
- Lets the model focus on learning task-specific nuances rather than general language patterns
- Often achieves higher accuracy on limited data due to transfer learning

For document classification, BERT's pre-trained representations of text already capture syntax, semantics, and world knowledge, so fine-tuning only needs to adapt those representations to your specific categories.